# SEC EDGAR 구조와 CIK

- company: Apple Inc.
- ticker: `AAPL`
- integer CIK: `320193`
- API path CIK: `0000320193`
- 확인 대상: 최근 10-k·10-Q 제출 이력, companyfacts JSON, `us-gaap` taxonomy, `Assets` facts
- 기준: XBRL fact는 `val`만 읽지 않고 unit, form, 기간, filed, accn을 함께 확인하다.
- 한계: companyfacts는 구조화된 facts이며 10-k·10-Q 원문과 주석 전체를 대신하지 않는다.

In [1]:
from getpass import getpass
from pathlib import Path
import time

import pandas as pd
import requests
from IPython.display import display


current_path = Path.cwd()
project_root = (
    current_path
    if (current_path / "notebooks").exists()
    else current_path.parent
)

notebooks_dir = project_root / "notebooks"
project_log_path = project_root / "docs" / "project_log.md"
previous_data_dictionary_path = (
    project_root / "docs" / "data_dictionary.md"
)
previous_chart_path = (
    project_root / "assets" / "macro_charts.png"
)
notebook_output_path = (
    notebooks_dir / "07_sec_structure_apple.ipynb"
)

required_paths = {
    "notebooks_dir": notebooks_dir,
    "project_log": project_log_path,
    "previous_data_dictionary": previous_data_dictionary_path,
    "previous_chart": previous_chart_path,
}

missing_path = [
    name
    for name, path in required_paths.items()
    if not path.exists()
]

if missing_path:
    raise FileNotFoundError(
        "준비 파일 또는 폴더가 없습니다: "
        f"{missing_path}"
    )

print("project_root:", project_root)
print("notebook_output_path:", notebook_output_path)
print("project_log_exists:", project_log_path.exists())
print(
    "previous_data_dictionary_exists:",
    previous_data_dictionary_path.exists()
)
print("previous_chart_exists:", previous_chart_path.exists())

project_root: /Users/im-youngchan/Desktop/US Financial
notebook_output_path: /Users/im-youngchan/Desktop/US Financial/notebooks/07_sec_structure_apple.ipynb
project_log_exists: True
previous_data_dictionary_exists: True
previous_chart_exists: True


In [2]:
SEC_TICKERS_URL = "https://www.sec.gov/files/company_tickers.json"
SEC_SUBMISSIONS_URL_TEMPLATE = (
    "https://data.sec.gov/submissions/CIK{cik}.json"
)
SEC_COMPANYFACTS_URL_TEMPLATE = (
    "https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
)

REQUEST_TIMEOUT_SECONDS = 30
REQUEST_INTERVAL_SECONDS = 0.2

print("request_timeout_seconds:", REQUEST_TIMEOUT_SECONDS)
print("request_interval_seconds:", REQUEST_INTERVAL_SECONDS)

request_timeout_seconds: 30
request_interval_seconds: 0.2


In [3]:
sec_contact_email = getpass(
    "SEC User-Agent에 넣을 연락처 이메일을 입력하세요: "
).strip()

if not sec_contact_email or "@" not in sec_contact_email:
    raise ValueError("연락 가능한 이메일 형식을 입력하세요.")

sec_user_agent = (
    f"U.s. Financial Research Porject {sec_contact_email}"
)
sec_headers = {
    "User-Agent": sec_user_agent,
    "Accept-Encoding": "gzip, deflate",
}

print("sec_user_agent_configured:", True)
print("sec_header_names:", list(sec_headers.keys()))

sec_user_agent_configured: True
sec_header_names: ['User-Agent', 'Accept-Encoding']


In [4]:
def get_sec_json(
        url,
        headers,
        pause_seconds=REQUEST_INTERVAL_SECONDS,
):
    if pause_seconds < 0:
        raise ValueError("pause_seconds는 이상이어야 합니다.")

    time.sleep(pause_seconds)

    response = requests.get(
        url,
        headers=headers,
        timeout=REQUEST_TIMEOUT_SECONDS,
    )
    response.raise_for_status()

    payload = response.json()

    if not isinstance(payload, dict):
        raise TypeError(
            "SEC JSON 최상위 구조가 딕셔너리가 아닙니다."
        )

    return payload

In [5]:
ticker_payload = get_sec_json(
    url=SEC_TICKERS_URL,
    headers=sec_headers,
)

ticker_df = pd.DataFrame(ticker_payload.values())

required_ticker_columns = {
    "cik_str",
    "ticker",
    "title",
}
missing_ticker_columns = (
    required_ticker_columns - set(ticker_df.columns)
)

if missing_ticker_columns:
    raise ValueError(
        "SEC ticker 목록에 필요한 column이 없습니다: "
        f"{sorted(missing_ticker_columns)}"
    )

ticker_df["ticker"] = ticker_df["ticker"].str.upper()

apple_ticker_df = ticker_df.loc[
    ticker_df["ticker"] == "AAPL",
    ["cik_str", "ticker", "title"],
].reset_index(drop=True)

if len(apple_ticker_df) != 1:
    raise ValueError(
        "SEC ticker 목록에서 AAPL을 하나의 row로 "
        "찾지 못했습니다."
    )

print("ticker_row_count:", len(ticker_df))
display(apple_ticker_df)

ticker_row_count: 10387


,cik_str,ticker,title
0,320193,AAPL,Apple Inc.


In [6]:
apple_cik_int = int(apple_ticker_df.loc[0, "cik_str"])
apple_cik = f"{apple_cik_int:010d}"
expected_apple_cik = "0000320193"

if apple_cik != expected_apple_cik:
    raise ValueError(
        f"Apple CIK가 예상과 다릅니다: {apple_cik}"
    )

print("apple_cik_int:", apple_cik_int)
print("apple_cik:", apple_cik)

apple_cik_int: 320193
apple_cik: 0000320193


In [7]:
submissions_url = SEC_SUBMISSIONS_URL_TEMPLATE.format(
    cik=apple_cik
)

apple_submissions_json = get_sec_json(
    url=submissions_url,
    headers=sec_headers,
)

if "filings" not in apple_submissions_json:
    raise KeyError("submissions JSON에서 filings를 찾지 못했습니다.")

if "recent" not in apple_submissions_json["filings"]:
    raise KeyError(
        "submissions JSON에서 filings → recent를 "
        "찾지 못했습니다."
    )

recent_filings_payload = apple_submissions_json["filings"]["recent"]
recent_filings_df = pd.DataFrame(recent_filings_payload)

required_filing_columns = [
    "filingDate",
    "reportDate",
    "form",
    "accessionNumber",
    "primaryDocument",
]
missing_filing_columns = [
    column
    for column in required_filing_columns
    if column not in recent_filings_df.columns
]

if missing_filing_columns:
    raise ValueError(
        "recent filings에 필요한 column이 없습니다: "
        f"{missing_filing_columns}"
    )

recent_filings_df["filingDate"] = pd.to_datetime(
    recent_filings_df["filingDate"],
    errors="coerce",
)
recent_filings_df["reportDate"] = pd.to_datetime(
    recent_filings_df["reportDate"],
    errors="coerce",
)

periodic_filings_df = (
    recent_filings_df.loc[
        recent_filings_df["form"].isin(["10-K", "10-Q"]),
        required_filing_columns,
    ]
    .sort_values("filingDate", ascending=False)
    .reset_index(drop=True)
)

if periodic_filings_df.empty:
    raise ValueError(
        "최근 filings에서 10-K 또는 10-Q를 찾지 못했습니다."
    )

print("submissions_top_level_keys:")
print(sorted(apple_submissions_json.keys()))
print("submissions_name:", apple_submissions_json.get("name"))
print("submissions_cik:", apple_submissions_json.get("cik"))
print("recent_filing_count:", len(recent_filings_df))
print("recent_10k_10q_count:", len(periodic_filings_df))
display(periodic_filings_df.head(10))

submissions_top_level_keys:
['addresses', 'category', 'cik', 'description', 'ein', 'entityType', 'exchanges', 'filings', 'fiscalYearEnd', 'flags', 'formerNames', 'insiderTransactionForIssuerExists', 'insiderTransactionForOwnerExists', 'investorWebsite', 'lei', 'name', 'ownerOrg', 'phone', 'sic', 'sicDescription', 'stateOfIncorporation', 'stateOfIncorporationDescription', 'tickers', 'website']
submissions_name: Apple Inc.
submissions_cik: 0000320193
recent_filing_count: 1001
recent_10k_10q_count: 45


,filingDate,reportDate,form,accessionNumber,primaryDocument
0,2026-07-31,2026-06-27,10-Q,0000320193-26-000020,aapl-20260627.htm
1,2026-05-01,2026-03-28,10-Q,0000320193-26-000013,aapl-20260328.htm
2,2026-01-30,2025-12-27,10-Q,0000320193-26-000006,aapl-20251227.htm
3,2025-10-31,2025-09-27,10-K,0000320193-25-000079,aapl-20250927.htm
4,2025-08-01,2025-06-28,10-Q,0000320193-25-000073,aapl-20250628.htm
5,2025-05-02,2025-03-29,10-Q,0000320193-25-000057,aapl-20250329.htm
6,2025-01-31,2024-12-28,10-Q,0000320193-25-000008,aapl-20241228.htm
7,2024-11-01,2024-09-28,10-K,0000320193-24-000123,aapl-20240928.htm
8,2024-08-02,2024-06-29,10-Q,0000320193-24-000081,aapl-20240629.htm
9,2024-05-03,2024-03-30,10-Q,0000320193-24-000069,aapl-20240330.htm


In [8]:
form_count = (
    periodic_filings_df["form"]
    .value_counts()
    .rename_axis("form")
    .reset_index(name="count")
)

print("periodic_form_count:")
display(form_count)

periodic_form_count:


,form,count
0,10-Q,34
1,10-K,11


In [9]:
companyfacts_url = SEC_COMPANYFACTS_URL_TEMPLATE.format(
    cik=apple_cik
)

apple_companyfacts_json = get_sec_json(
    url=companyfacts_url,
    headers=sec_headers,
)

required_companyfacts_keys = {
    "cik",
    "entityName",
    "facts",
}
missing_companyfacts_keys = (
    required_companyfacts_keys
    - set(apple_companyfacts_json.keys())
)

if missing_companyfacts_keys:
    raise ValueError(
        "companyfacts 최상위 key가 부족합니다: "
        f"{sorted(missing_companyfacts_keys)}"
    )

facts_by_taxonomy = apple_companyfacts_json["facts"]
taxonomy_names = sorted(facts_by_taxonomy.keys())

if "us-gaap" not in facts_by_taxonomy:
    raise KeyError(
        "compnayfacts에서 us-gaap taxonomy를 찾지 못했습니다."
    )

us_gaap_facts = facts_by_taxonomy["us-gaap"]

print("companyfacts_top_level_keys:")
print(sorted(apple_companyfacts_json.keys()))
print("companyfacts_cik:", apple_companyfacts_json["cik"])
print(
    "companyfacts_entity_name:",
    apple_companyfacts_json["entityName"],
)
print("taxonomy_names:", taxonomy_names)
print("us_gaap_concept_count:", len(us_gaap_facts))

companyfacts_top_level_keys:
['cik', 'entityName', 'facts']
companyfacts_cik: 320193
companyfacts_entity_name: Apple Inc.
taxonomy_names: ['dei', 'us-gaap']
us_gaap_concept_count: 503


In [10]:
concept_preview_rows = []

for concept_id in sorted(us_gaap_facts.keys())[:20]:
    concept_data = us_gaap_facts[concept_id]
    concept_preview_rows.append(
        {
            "concept_id": concept_id,
            "label": concept_data.get("label"),
            "unit_names": ", ".join(
                sorted(concept_data.get("units", {}).keys())
            ),
        }
    )

concept_preview_df = pd.DataFrame(concept_preview_rows)

assets_concept_id = "Assets"
assets_concept = us_gaap_facts.get(assets_concept_id)

if assets_concept is None:
    raise KeyError(
        "us-gaap taxonomy에서 Assets concept를 찾지 못했습니다."
    )

assets_units = assets_concept.get("units", {})
assets_unit_names = sorted(assets_units.keys())

if "USD" not in assets_units:
    raise KeyError("Assets concept에서 USD unit을 찾지 못했습니다.")

print("concept_preview_row_count:", len(concept_preview_df))
display(concept_preview_df)

print("assets_concept_id:", assets_concept_id)
print("assets_label:", assets_concept.get("label"))
print("assets_description:", assets_concept.get("description"))
print("assets_unit_names:", assets_unit_names)

concept_preview_row_count: 20


,concept_id,label,unit_names
0,AccountsPayable,Accounts Payable (Deprecated 2009-01-31),USD
1,AccountsPayableCurrent,"Accounts Payable, Current",USD
2,AccountsReceivableNetCurrent,"Accounts Receivable, after Allowance for Credi...",USD
3,AccruedIncomeTaxesCurrent,"Accrued Income Taxes, Current",USD
4,AccruedIncomeTaxesNoncurrent,"Accrued Income Taxes, Noncurrent",USD
5,AccruedLiabilities,Accrued Liabilities (Deprecated 2009-01-31),USD
6,AccruedLiabilitiesCurrent,"Accrued Liabilities, Current",USD
7,AccruedMarketingCostsCurrent,"Accrued Marketing Costs, Current",USD
8,AccumulatedDepreciationDepletionAndAmortizatio...,"Accumulated Depreciation, Depletion and Amorti...",USD
9,AccumulatedOtherComprehensiveIncomeLossAvailab...,"AOCI, Debt Securities, Available-for-sale, Adj...",USD


assets_concept_id: Assets
assets_label: Assets
assets_description: Sum of the carrying amounts as of the balance sheet date of all assets that are recognized. Assets are probable future economic benefits obtained or controlled by an entity as a result of past transactions or events.
assets_unit_names: ['USD']


In [18]:
assets_facts_df = pd.DataFrame(assets_units["USD"])

if assets_facts_df.empty:
    raise ValueError("Assets의 USD fact record가 비어 있습니다.")

required_assets_columns = {
    "end",
    "val", 
    "form",
    "filed", 
    "accn",
}
missing_assets_columns = (
    required_assets_columns - set(assets_facts_df.columns)
)

if missing_assets_columns:
    raise ValueError(
        "Assets facts에 필요한 column이 없습니다: "
        f"{sorted(missing_assets_columns)}"
    )

assets_facts_df["end"] = pd.to_datetime(
    assets_facts_df["end"],
    errors="coerce",
)
assets_facts_df["filed"] = pd.to_datetime(
    assets_facts_df["filed"],
    errors="coerce",
)
assets_facts_df["val"] = pd.to_numeric(
    assets_facts_df["val"],
    errors="coerce",
)

assets_periodic_df = (
    assets_facts_df.loc[
        assets_facts_df["form"].isin(["10-K", "10-Q"])
    ]
    .sort_values(
        ["end", "filed"], 
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

if assets_periodic_df.empty:
    raise ValueError(
        "Assets facts에서 10-K·10-Q record를 찾지 못했습니다."
    )

fact_display_columns = [
    "end",
    "val",
    "form",
    "fy",
    "fp",
    "filed",
    "accn",
    "frame",
]
available_fact_columns = [
    column
    for column in fact_display_columns
    if column in assets_periodic_df.columns
]

repeated_end_mask = assets_periodic_df.duplicated(
    subset=["end"],
    keep=False,
)
repeated_end_df = assets_periodic_df.loc[
    repeated_end_mask,
    available_fact_columns,
].sort_values(
    ["end", "filed"],
    ascending=[False, False],
)

print("assets_fact_count:", len(assets_facts_df))
print("assets_columns:", list(assets_facts_df.columns))
print(
    "assets_missing_end_count:",
    assets_facts_df["end"].isna().sum(),
)
print(
    "assets_missing_val_count:",
    assets_facts_df["val"].isna().sum(),
)
print("assets_10k_10q_count:", len(assets_periodic_df))
print("repeated_end_record_count:", len(repeated_end_df))

display(assets_periodic_df[available_fact_columns].head(15))
display(repeated_end_df.head(20))

assets_fact_count: 146
assets_columns: ['end', 'val', 'accn', 'fy', 'fp', 'form', 'filed', 'frame']
assets_missing_end_count: 0
assets_missing_val_count: 0
assets_10k_10q_count: 140
repeated_end_record_count: 88


,end,val,form,fy,fp,filed,accn,frame
0,2026-06-27,383266000000,10-Q,2026.0,Q3,2026-07-31,0000320193-26-000020,CY2026Q2I
1,2026-03-28,371082000000,10-Q,2026.0,Q2,2026-05-01,0000320193-26-000013,CY2026Q1I
2,2025-12-27,379297000000,10-Q,2026.0,Q1,2026-01-30,0000320193-26-000006,CY2025Q4I
3,2025-09-27,359241000000,10-Q,2026.0,Q3,2026-07-31,0000320193-26-000020,CY2025Q3I
4,2025-09-27,359241000000,10-Q,2026.0,Q2,2026-05-01,0000320193-26-000013,NaN
5,2025-09-27,359241000000,10-Q,2026.0,Q1,2026-01-30,0000320193-26-000006,NaN
6,2025-09-27,359241000000,10-K,2025.0,FY,2025-10-31,0000320193-25-000079,NaN
7,2025-06-28,331495000000,10-Q,2025.0,Q3,2025-08-01,0000320193-25-000073,CY2025Q2I
8,2025-03-29,331233000000,10-Q,2025.0,Q2,2025-05-02,0000320193-25-000057,CY2025Q1I
9,2024-12-28,344085000000,10-Q,2025.0,Q1,2025-01-31,0000320193-25-000008,CY2024Q4I


,end,val,form,fy,fp,filed,accn,frame
3,2025-09-27,359241000000,10-Q,2026.0,Q3,2026-07-31,0000320193-26-000020,CY2025Q3I
4,2025-09-27,359241000000,10-Q,2026.0,Q2,2026-05-01,0000320193-26-000013,NaN
5,2025-09-27,359241000000,10-Q,2026.0,Q1,2026-01-30,0000320193-26-000006,NaN
6,2025-09-27,359241000000,10-K,2025.0,FY,2025-10-31,0000320193-25-000079,NaN
10,2024-09-28,364980000000,10-K,2025.0,FY,2025-10-31,0000320193-25-000079,CY2024Q3I
11,2024-09-28,364980000000,10-Q,2025.0,Q3,2025-08-01,0000320193-25-000073,NaN
12,2024-09-28,364980000000,10-Q,2025.0,Q2,2025-05-02,0000320193-25-000057,NaN
13,2024-09-28,364980000000,10-Q,2025.0,Q1,2025-01-31,0000320193-25-000008,NaN
14,2024-09-28,364980000000,10-K,2024.0,FY,2024-11-01,0000320193-24-000123,NaN
18,2023-09-30,352583000000,10-K,2024.0,FY,2024-11-01,0000320193-24-000123,CY2023Q3I


In [19]:
revenue_keywords = ("Revenue", "Sales")
revenue_candidate_ids = sorted(
    concept_id
    for concept_id in us_gaap_facts
    if any(
        keyword in concept_id
        for keyword in revenue_keywords
    )
)

revenue_candidate_rows = []

for concept_id in revenue_candidate_ids:
    concept_data = us_gaap_facts[concept_id]
    revenue_candidate_rows.append(
        {
            "concept_id": concept_id,
            "label": concept_data.get("label"),
            "description": concept_data.get("description"),
            "unit_names": ", ".join(
                sorted(concept_data.get("units", {}).keys())
            ),
        }
    )

revenue_candidate_df = pd.DataFrame(revenue_candidate_rows)

print("revenue_candidate_count:", len(revenue_candidate_df))
display(revenue_candidate_df.head(20))

revenue_candidate_count: 8


,concept_id,label,description,unit_names
0,ContractWithCustomerLiabilityRevenueRecognized,"Contract with Customer, Liability, Revenue Rec...",Amount of revenue recognized that was previous...,USD
1,DeferredRevenueCurrent,"Deferred Revenue, Current",Amount of deferred income and obligation to tr...,USD
2,DeferredRevenueNoncurrent,"Deferred Revenue, Noncurrent",Amount of deferred income and obligation to tr...,USD
3,IncreaseDecreaseInDeferredRevenue,Increase (Decrease) in Deferred Revenue,Amount of increase (decrease) in deferred inco...,USD
4,RevenueFromContractWithCustomerExcludingAssess...,"Revenue from Contract with Customer, Excluding...","Amount, excluding tax collected from customer,...",USD
5,Revenues,Revenues,"Amount of revenue recognized from goods sold, ...",USD
6,SalesRevenueNet,"Revenue, Net (Deprecated 2018-01-31)",Total revenue from sale of goods and services ...,USD
7,SalesRevenueServicesGross,"Sales Revenue, Services, Other (Deprecated 201...",Amount before allowances and discounts of serv...,USD


In [20]:
apple_structure_summary = {
    "ticker": "AAPL",
    "cik_integer": apple_cik_int,
    "cik_10_digit": apple_cik,
    "entity_name": apple_companyfacts_json["entityName"],
    "taxonomy_names": ", ".join(taxonomy_names),
    "us_gaap_concept_count": len(us_gaap_facts),
    "recent_10k_10q_count": len(periodic_filings_df),
    "assets_unit_names": ", ".join(assets_unit_names),
    "assets_fact_count": len(assets_facts_df),
    "revenue_candidate_count": len(revenue_candidate_df),
}

apple_structure_summary_df = (
    pd.Series(apple_structure_summary, name="value")
    .rename_axis("item")
    .reset_index()
)

display(apple_structure_summary_df)

,item,value
0,ticker,AAPL
1,cik_integer,320193
2,cik_10_digit,0000320193
3,entity_name,Apple Inc.
4,taxonomy_names,"dei, us-gaap"
5,us_gaap_concept_count,503
6,recent_10k_10q_count,45
7,assets_unit_names,USD
8,assets_fact_count,146
9,revenue_candidate_count,8
